# Integrasi Data Terstruktur dan Tidak Terstruktur untuk Data-driven Decision Making

**Studi kasus:** Brazilian E-Commerce Public Dataset by Olist  
**Platform:** Google Colab  
**Program studi:** S2 Bisnis Digital — Universitas Dinamika  
**Nama:** Lefi Andri Lestari  
**NIM:** 25630300009

Notebook ini mengintegrasikan data transaksi, pelanggan, produk, pembayaran, pengiriman, rating, dan teks ulasan pada tingkat transaksi (`order_id`) serta pelanggan (`customer_unique_id`).

## Tujuan

1. Menggabungkan data terstruktur dan teks ulasan tanpa menggandakan transaksi.
2. Menjelaskan kinerja penjualan, pengiriman, dan kepuasan pelanggan.
3. Menguji hubungan keterlambatan dan ongkos kirim dengan kepuasan.
4. Mengidentifikasi sentimen dan topik keluhan dari teks ulasan.
5. Membuat model early-warning untuk memprediksi ulasan negatif.
6. Mengubah hasil analisis menjadi rekomendasi keputusan bisnis.

> Jalankan menu **Runtime → Run all**. Proses pertama dapat memerlukan beberapa menit karena mengunduh dan memproses data.


## Kerangka Data-driven Decision Making

| Tahap | Pertanyaan | Luaran notebook |
|---|---|---|
| Descriptive | Apa yang terjadi? | KPI, tren pesanan, rating, keterlambatan |
| Diagnostic | Mengapa terjadi? | Uji statistik, sentimen, topik keluhan |
| Predictive | Apa yang mungkin terjadi? | Model risiko ulasan negatif |
| Prescriptive | Apa yang sebaiknya dilakukan? | Prioritas tindakan bisnis |

**Unit analisis utama:** satu baris mewakili satu pesanan. Tabel item dan pembayaran yang bersifat *one-to-many* harus diagregasi sebelum digabungkan.


## 1. Persiapan lingkungan Google Colab


In [ ]:
# Menginstal pustaka yang mungkin belum tersedia di Google Colab.
# Opsi -q mengurangi keluaran instalasi agar notebook lebih rapi.
!pip -q install unidecode wordcloud openpyxl


In [ ]:
import os
import re
import json
import math
import shutil
import warnings
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from IPython.display import display, Markdown
from scipy import stats
from scipy.stats import chi2_contingency, mannwhitneyu, spearmanr
from unidecode import unidecode
from wordcloud import WordCloud

import nltk
from nltk.corpus import stopwords

from sklearn.compose import ColumnTransformer
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.decomposition import NMF
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    accuracy_score, classification_report, confusion_matrix,
    f1_score, precision_score, recall_score, roc_auc_score,
    ConfusionMatrixDisplay
)
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler

warnings.filterwarnings("ignore")
pd.set_option("display.max_columns", 100)
pd.set_option("display.float_format", lambda x: f"{x:,.3f}")
sns.set_theme(style="whitegrid", palette="Set2")
plt.rcParams["figure.figsize"] = (10, 5)
plt.rcParams["axes.titlesize"] = 13
plt.rcParams["axes.labelsize"] = 11

RANDOM_STATE = 42
NLP_SAMPLE_MAX = 40_000  # Turunkan menjadi 15_000 jika RAM terbatas.
TOP_N_CATEGORIES = 12
N_TOPICS = 5

nltk.download("stopwords", quiet=True)
PORTUGUESE_STOPWORDS = set(stopwords.words("portuguese"))

print("Lingkungan analisis siap.")


## 2. Mengunduh dataset

Notebook akan mencoba mengunduh dataset publik melalui KaggleHub. Jika unduhan otomatis gagal, unggah file ZIP yang sudah diunduh dari:

https://www.kaggle.com/datasets/olistbr/brazilian-ecommerce


In [ ]:
DATA_ROOT = Path("/content/olist_data")
DATA_ROOT.mkdir(parents=True, exist_ok=True)

try:
    import kagglehub
    downloaded_path = Path(kagglehub.dataset_download("olistbr/brazilian-ecommerce"))
    print(f"Dataset berhasil diunduh ke: {downloaded_path}")
    SEARCH_ROOT = downloaded_path
except Exception as exc:
    print("Unduhan otomatis gagal:", exc)
    print("Silakan unggah ZIP dataset Olist dari Kaggle.")
    from google.colab import files
    uploaded = files.upload()
    zip_names = [name for name in uploaded if name.lower().endswith(".zip")]
    if not zip_names:
        raise ValueError("Tidak ditemukan file ZIP. Unduh lalu unggah ZIP dataset Olist.")
    with zipfile.ZipFile(zip_names[0], "r") as archive:
        archive.extractall(DATA_ROOT)
    SEARCH_ROOT = DATA_ROOT

csv_files = list(Path(SEARCH_ROOT).rglob("*.csv"))
print(f"Jumlah file CSV ditemukan: {len(csv_files)}")
for file in sorted(csv_files):
    print("-", file.name)


## 3. Memuat seluruh tabel dan mengenali jenis datanya


In [ ]:
STRUCTURED_FILES = {
    "customers": "olist_customers_dataset.csv",
    "geolocation": "olist_geolocation_dataset.csv",
    "items": "olist_order_items_dataset.csv",
    "payments": "olist_order_payments_dataset.csv",
    "orders": "olist_orders_dataset.csv",
    "products": "olist_products_dataset.csv",
    "sellers": "olist_sellers_dataset.csv",
    "category_translation": "product_category_name_translation.csv",
}

UNSTRUCTURED_FILES = {
    "reviews": "olist_order_reviews_dataset.csv",
}

file_paths = {
    **{name: STRUCTURED_DIR / filename for name, filename in STRUCTURED_FILES.items()},
    **{name: UNSTRUCTURED_DIR / filename for name, filename in UNSTRUCTURED_FILES.items()},
}

missing_files = [str(path) for path in file_paths.values() if not path.is_file()]
if missing_files:
    raise FileNotFoundError("File wajib tidak ditemukan:\n- " + "\n- ".join(missing_files))

tables = {
    name: pd.read_csv(path, low_memory=False)
    for name, path in file_paths.items()
}

for name, df in tables.items():
    source_type = "tidak terstruktur" if name in UNSTRUCTURED_FILES else "terstruktur"
    print(f"{name:22s}: {df.shape[0]:>9,} baris × {df.shape[1]:>2} kolom | {source_type}")


In [ ]:
data_classification = pd.DataFrame([
    ["olist_customers_dataset.csv", "Terstruktur", "data-terstruktur", "Identitas dan lokasi pelanggan"],
    ["olist_geolocation_dataset.csv", "Terstruktur", "data-terstruktur", "Koordinat dan wilayah"],
    ["olist_order_items_dataset.csv", "Terstruktur", "data-terstruktur", "Produk, seller, harga, ongkir"],
    ["olist_order_payments_dataset.csv", "Terstruktur", "data-terstruktur", "Metode dan nilai pembayaran"],
    ["olist_orders_dataset.csv", "Terstruktur", "data-terstruktur", "Status serta timestamp pesanan"],
    ["olist_products_dataset.csv", "Terstruktur", "data-terstruktur", "Kategori dan atribut produk"],
    ["olist_sellers_dataset.csv", "Terstruktur", "data-terstruktur", "Identitas dan lokasi seller"],
    ["product_category_name_translation.csv", "Terstruktur", "data-terstruktur", "Terjemahan kategori"],
    ["olist_order_reviews_dataset.csv", "Tidak terstruktur + metadata", "data-tidak-terstruktur", "Rating sebagai metadata; judul dan pesan sebagai teks ulasan"],
], columns=["file", "jenis_data", "folder_sumber", "isi"])
display(data_classification)

print("Kolom pada tabel ulasan:")
display(tables["reviews"].head(3))


## 4. Pemeriksaan kualitas data

Pemeriksaan awal mencakup duplikasi, nilai kosong, dan tipe data. Nilai kosong pada teks ulasan tidak otomatis berarti transaksi tidak valid: sebagian pelanggan hanya memberi rating tanpa menulis komentar.


In [ ]:
def quality_report(name, df):
    return pd.DataFrame({
        "table": name,
        "column": df.columns,
        "dtype": df.dtypes.astype(str).values,
        "missing_n": df.isna().sum().values,
        "missing_pct": (df.isna().mean().values * 100).round(2),
        "unique_n": [df[col].nunique(dropna=True) for col in df.columns],
    })

quality = pd.concat(
    [quality_report(name, df) for name, df in tables.items()],
    ignore_index=True
)
display(quality.sort_values(["missing_pct", "table"], ascending=[False, True]).head(30))

duplicates = pd.DataFrame({
    "table": tables.keys(),
    "duplicate_rows": [df.duplicated().sum() for df in tables.values()]
})
display(duplicates)


## 5. Pembersihan dan agregasi sebelum penggabungan

**Aturan penting:**

- Tabel item mempunyai beberapa baris untuk satu `order_id`.
- Tabel pembayaran dapat mempunyai beberapa metode untuk satu `order_id`.
- Tabel review perlu dikonsolidasikan apabila ditemukan lebih dari satu catatan per pesanan.

Karena itu, ketiga tabel tersebut diagregasi terlebih dahulu sehingga hasil akhirnya tetap satu baris per pesanan.


In [ ]:
customers = tables["customers"].copy()
items = tables["items"].copy()
payments = tables["payments"].copy()
reviews = tables["reviews"].copy()
orders = tables["orders"].copy()
products = tables["products"].copy()
sellers = tables["sellers"].copy()
category_translation = tables["category_translation"].copy()

# Tipe tanggal.
order_date_columns = [
    "order_purchase_timestamp", "order_approved_at",
    "order_delivered_carrier_date", "order_delivered_customer_date",
    "order_estimated_delivery_date"
]
for col in order_date_columns:
    orders[col] = pd.to_datetime(orders[col], errors="coerce")

for col in ["shipping_limit_date"]:
    items[col] = pd.to_datetime(items[col], errors="coerce")

for col in ["review_creation_date", "review_answer_timestamp"]:
    reviews[col] = pd.to_datetime(reviews[col], errors="coerce")

# Menghapus duplikasi baris identik, tetapi tidak menghapus catatan one-to-many yang valid.
for frame in [customers, items, payments, reviews, orders, products, sellers, category_translation]:
    frame.drop_duplicates(inplace=True)

# Menambahkan terjemahan kategori produk.
products = products.merge(category_translation, on="product_category_name", how="left")
products["product_category"] = products["product_category_name_english"].fillna(
    products["product_category_name"]
).fillna("unknown")

print("Pembersihan dasar selesai.")


In [ ]:
# ---------- Agregasi item pada tingkat order_id ----------
item_detail = items.merge(
    products[["product_id", "product_category"]],
    on="product_id", how="left"
)
item_detail["product_category"] = item_detail["product_category"].fillna("unknown")

item_agg = item_detail.groupby("order_id", as_index=False).agg(
    item_value=("price", "sum"),
    freight_total=("freight_value", "sum"),
    item_count=("order_item_id", "count"),
    product_count=("product_id", "nunique"),
    seller_count=("seller_id", "nunique"),
)

# Kategori dominan = kategori dengan nilai produk terbesar dalam pesanan.
category_value = (
    item_detail.groupby(["order_id", "product_category"], as_index=False)["price"]
    .sum()
    .sort_values(["order_id", "price"], ascending=[True, False])
)
dominant_category = (
    category_value.drop_duplicates("order_id")
    .rename(columns={"product_category": "dominant_category"})
    [["order_id", "dominant_category"]]
)
item_agg = item_agg.merge(dominant_category, on="order_id", how="left")

# ---------- Agregasi pembayaran pada tingkat order_id ----------
def mode_or_first(series):
    mode = series.mode(dropna=True)
    return mode.iloc[0] if not mode.empty else (series.dropna().iloc[0] if series.notna().any() else np.nan)

payment_agg = payments.groupby("order_id", as_index=False).agg(
    payment_total=("payment_value", "sum"),
    payment_installments=("payment_installments", "max"),
    payment_method_count=("payment_type", "nunique"),
    primary_payment_type=("payment_type", mode_or_first),
)

# ---------- Agregasi review pada tingkat order_id ----------
reviews["review_comment_title"] = reviews["review_comment_title"].fillna("").astype(str)
reviews["review_comment_message"] = reviews["review_comment_message"].fillna("").astype(str)
reviews["review_text"] = (
    reviews["review_comment_title"].str.strip() + " " +
    reviews["review_comment_message"].str.strip()
).str.replace(r"\s+", " ", regex=True).str.strip()

def join_nonempty(series):
    values = [str(v).strip() for v in series if str(v).strip()]
    return " || ".join(dict.fromkeys(values))

review_agg = reviews.groupby("order_id", as_index=False).agg(
    review_score=("review_score", "mean"),
    review_text=("review_text", join_nonempty),
    review_count=("review_id", "nunique"),
    review_creation_date=("review_creation_date", "min"),
    review_answer_timestamp=("review_answer_timestamp", "max"),
)

print("Ukuran tabel hasil agregasi:")
print("item_agg   :", item_agg.shape)
print("payment_agg:", payment_agg.shape)
print("review_agg :", review_agg.shape)


## 6. Penggabungan data pada tingkat transaksi


In [ ]:
order_base = orders.merge(customers, on="customer_id", how="left", validate="many_to_one")
integrated = (
    order_base
    .merge(item_agg, on="order_id", how="left", validate="one_to_one")
    .merge(payment_agg, on="order_id", how="left", validate="one_to_one")
    .merge(review_agg, on="order_id", how="left", validate="one_to_one")
)

# Memastikan unit analisis tetap satu baris per transaksi.
assert integrated["order_id"].is_unique, "order_id tidak unik setelah penggabungan."
assert len(integrated) == orders["order_id"].nunique(), "Jumlah transaksi berubah setelah merge."

# Feature engineering.
integrated["delivery_days"] = (
    integrated["order_delivered_customer_date"] - integrated["order_purchase_timestamp"]
).dt.total_seconds() / 86400
integrated["delivery_delay_days"] = (
    integrated["order_delivered_customer_date"] - integrated["order_estimated_delivery_date"]
).dt.total_seconds() / 86400
integrated["is_late"] = np.where(
    integrated["delivery_delay_days"].notna(),
    (integrated["delivery_delay_days"] > 0).astype(int),
    np.nan
)
integrated["total_order_cost"] = integrated["item_value"].fillna(0) + integrated["freight_total"].fillna(0)
integrated["freight_ratio"] = np.where(
    integrated["item_value"] > 0,
    integrated["freight_total"] / integrated["item_value"],
    np.nan
)
integrated["purchase_month"] = integrated["order_purchase_timestamp"].dt.to_period("M").astype(str)
integrated["purchase_year"] = integrated["order_purchase_timestamp"].dt.year
integrated["has_review_text"] = integrated["review_text"].fillna("").str.strip().ne("")

# Label sentimen berbasis rating sebagai label terukur, bukan hasil tebakan manual.
def rating_to_sentiment(score):
    if pd.isna(score):
        return np.nan
    if score <= 2:
        return "Negatif"
    # Nilai dapat pecahan jika satu order memiliki lebih dari satu review.
    if score < 4:
        return "Netral"
    return "Positif"

integrated["rating_sentiment"] = integrated["review_score"].apply(rating_to_sentiment)
integrated["negative_review"] = np.where(
    integrated["review_score"].notna(),
    (integrated["review_score"] <= 2).astype(int),
    np.nan
)

print(f"Dataset terintegrasi: {len(integrated):,} transaksi × {integrated.shape[1]} kolom")
print("order_id unik:", integrated["order_id"].is_unique)
display(integrated.head(3))


In [ ]:
# Ringkasan kelengkapan variabel utama.
key_columns = [
    "order_id", "customer_unique_id", "item_value", "freight_total",
    "payment_total", "delivery_days", "delivery_delay_days",
    "review_score", "review_text", "dominant_category"
]
completeness = pd.DataFrame({
    "column": key_columns,
    "available_n": [integrated[col].notna().sum() for col in key_columns],
    "missing_n": [integrated[col].isna().sum() for col in key_columns],
    "missing_pct": [(integrated[col].isna().mean() * 100) for col in key_columns],
}).round(2)
display(completeness)


## 7. Descriptive analytics: apa yang terjadi?


In [ ]:
delivered = integrated[integrated["order_status"].eq("delivered")].copy()
reviewed = integrated[integrated["review_score"].notna()].copy()

kpis = {
    "Jumlah pesanan": integrated["order_id"].nunique(),
    "Jumlah pelanggan unik": integrated["customer_unique_id"].nunique(),
    "Total nilai produk": integrated["item_value"].sum(),
    "Rata-rata nilai pesanan": integrated["item_value"].mean(),
    "Rata-rata rating": reviewed["review_score"].mean(),
    "Persentase pesanan terlambat": delivered["is_late"].mean() * 100,
    "Persentase review negatif": reviewed["negative_review"].mean() * 100,
    "Ulasan dengan teks": integrated["has_review_text"].sum(),
}
display(pd.DataFrame({"KPI": kpis.keys(), "Nilai": kpis.values()}))


In [ ]:
monthly = (
    integrated.dropna(subset=["order_purchase_timestamp"])
    .set_index("order_purchase_timestamp")
    .resample("MS")
    .agg(orders=("order_id", "nunique"), revenue=("item_value", "sum"))
    .reset_index()
)

fig, axes = plt.subplots(1, 2, figsize=(16, 5))
sns.lineplot(data=monthly, x="order_purchase_timestamp", y="orders", marker="o", ax=axes[0])
axes[0].set_title("Tren Jumlah Pesanan Bulanan")
axes[0].set_xlabel("Bulan")
axes[0].set_ylabel("Jumlah pesanan")

sns.lineplot(data=monthly, x="order_purchase_timestamp", y="revenue", marker="o", ax=axes[1])
axes[1].set_title("Tren Nilai Produk Bulanan")
axes[1].set_xlabel("Bulan")
axes[1].set_ylabel("Nilai produk")
plt.tight_layout()
plt.show()


In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

score_counts = reviewed["review_score"].round().value_counts().sort_index()
sns.barplot(x=score_counts.index.astype(str), y=score_counts.values, ax=axes[0])
axes[0].set_title("Distribusi Rating")
axes[0].set_xlabel("Rating")
axes[0].set_ylabel("Jumlah ulasan")

sentiment_counts = reviewed["rating_sentiment"].value_counts().reindex(["Negatif", "Netral", "Positif"])
sns.barplot(x=sentiment_counts.index, y=sentiment_counts.values, ax=axes[1])
axes[1].set_title("Sentimen Berdasarkan Rating")
axes[1].set_xlabel("Sentimen")
axes[1].set_ylabel("Jumlah ulasan")

late_rating = delivered.dropna(subset=["review_score", "is_late"]).copy()
late_rating["status_delivery"] = late_rating["is_late"].map({0: "Tepat/lebih awal", 1: "Terlambat"})
sns.boxplot(data=late_rating, x="status_delivery", y="review_score", ax=axes[2])
axes[2].set_title("Rating menurut Ketepatan Pengiriman")
axes[2].set_xlabel("")
axes[2].set_ylabel("Rating")

plt.tight_layout()
plt.show()


In [ ]:
category_summary = (
    reviewed.groupby("dominant_category", dropna=False)
    .agg(
        orders=("order_id", "nunique"),
        revenue=("item_value", "sum"),
        avg_rating=("review_score", "mean"),
        negative_rate=("negative_review", "mean"),
        late_rate=("is_late", "mean"),
    )
    .query("orders >= 30")
    .sort_values("orders", ascending=False)
)

display(category_summary.head(15).style.format({
    "revenue": "{:,.2f}", "avg_rating": "{:.2f}",
    "negative_rate": "{:.1%}", "late_rate": "{:.1%}"
}))

top_categories = category_summary.head(TOP_N_CATEGORIES).sort_values("negative_rate")
plt.figure(figsize=(10, 6))
sns.barplot(data=top_categories.reset_index(), x="negative_rate", y="dominant_category")
plt.title("Tingkat Ulasan Negatif pada Kategori dengan Pesanan Terbanyak")
plt.xlabel("Proporsi ulasan negatif")
plt.ylabel("Kategori dominan")
plt.gca().xaxis.set_major_formatter(plt.matplotlib.ticker.PercentFormatter(1.0))
plt.tight_layout()
plt.show()


## 8. Analisis pelanggan: RFM dan pembelian berulang

`customer_unique_id` digunakan karena dapat mewakili pelanggan yang sama pada beberapa pesanan. RFM terdiri dari:

- **Recency:** jarak hari sejak pembelian terakhir.
- **Frequency:** jumlah pesanan pelanggan.
- **Monetary:** total nilai pembelian.


In [ ]:
analysis_date = integrated["order_purchase_timestamp"].max() + pd.Timedelta(days=1)

customer_summary = (
    integrated.dropna(subset=["customer_unique_id"])
    .groupby("customer_unique_id", as_index=False)
    .agg(
        last_purchase=("order_purchase_timestamp", "max"),
        frequency=("order_id", "nunique"),
        monetary=("item_value", "sum"),
        avg_rating=("review_score", "mean"),
        negative_review_rate=("negative_review", "mean"),
        late_rate=("is_late", "mean"),
    )
)
customer_summary["recency_days"] = (analysis_date - customer_summary["last_purchase"]).dt.days
customer_summary["customer_type"] = np.where(
    customer_summary["frequency"] > 1, "Pelanggan berulang", "Satu kali pembelian"
)

repeat_summary = customer_summary.groupby("customer_type").agg(
    customers=("customer_unique_id", "nunique"),
    avg_frequency=("frequency", "mean"),
    avg_monetary=("monetary", "mean"),
    avg_rating=("avg_rating", "mean"),
    avg_negative_rate=("negative_review_rate", "mean"),
)
display(repeat_summary.style.format({
    "avg_frequency": "{:.2f}", "avg_monetary": "{:,.2f}",
    "avg_rating": "{:.2f}", "avg_negative_rate": "{:.1%}"
}))

plt.figure(figsize=(8, 4))
sns.countplot(data=customer_summary, x="customer_type")
plt.title("Komposisi Pelanggan Berdasarkan Frekuensi Pembelian")
plt.xlabel("")
plt.ylabel("Jumlah pelanggan")
plt.tight_layout()
plt.show()


## 9. Diagnostic analytics: uji statistik

Tingkat signifikansi yang digunakan adalah $\alpha = 0{,}05$.

1. **Spearman:** hubungan keterlambatan dengan rating.
2. **Mann–Whitney:** perbedaan rating pesanan terlambat dan tepat waktu.
3. **Chi-square dan Cramér's V:** hubungan kategori produk dengan ulasan negatif.

Uji nonparametrik digunakan karena rating bersifat ordinal dan tidak harus berdistribusi normal.


In [ ]:
statistical_results = []

test_data = delivered.dropna(subset=["delivery_delay_days", "review_score"]).copy()

# Uji 1: Spearman.
rho, p_spearman = spearmanr(test_data["delivery_delay_days"], test_data["review_score"])
statistical_results.append({
    "test": "Spearman: keterlambatan vs rating",
    "statistic": rho,
    "p_value": p_spearman,
    "significant": p_spearman < 0.05,
    "interpretation": "Hubungan signifikan" if p_spearman < 0.05 else "Belum terdapat bukti hubungan signifikan"
})

# Uji 2: Mann–Whitney.
late_scores = test_data.loc[test_data["is_late"] == 1, "review_score"]
ontime_scores = test_data.loc[test_data["is_late"] == 0, "review_score"]
u_stat, p_mann = mannwhitneyu(late_scores, ontime_scores, alternative="two-sided")
statistical_results.append({
    "test": "Mann–Whitney: rating terlambat vs tepat waktu",
    "statistic": u_stat,
    "p_value": p_mann,
    "significant": p_mann < 0.05,
    "interpretation": "Rating kedua kelompok berbeda signifikan" if p_mann < 0.05 else "Belum terdapat bukti perbedaan signifikan"
})

# Uji 3: Chi-square pada kategori dengan volume tertinggi.
category_pool = reviewed["dominant_category"].value_counts().head(TOP_N_CATEGORIES).index
chi_data = reviewed[reviewed["dominant_category"].isin(category_pool)].dropna(subset=["negative_review"])
contingency = pd.crosstab(chi_data["dominant_category"], chi_data["negative_review"])
chi2, p_chi, dof, expected = chi2_contingency(contingency)
n_chi = contingency.to_numpy().sum()
min_dim = min(contingency.shape) - 1
cramers_v = math.sqrt(chi2 / (n_chi * min_dim)) if min_dim > 0 else np.nan
statistical_results.append({
    "test": "Chi-square: kategori vs ulasan negatif",
    "statistic": chi2,
    "p_value": p_chi,
    "significant": p_chi < 0.05,
    "interpretation": f"Cramér's V = {cramers_v:.3f}"
})

stats_table = pd.DataFrame(statistical_results)
display(stats_table.style.format({"statistic": "{:,.4f}", "p_value": "{:.6g}"}))

print(f"Rata-rata rating tepat/lebih awal: {ontime_scores.mean():.3f}")
print(f"Rata-rata rating terlambat      : {late_scores.mean():.3f}")
print(f"Selisih rating                  : {ontime_scores.mean() - late_scores.mean():.3f}")


## 10. Pengolahan data tidak terstruktur

Ulasan Olist menggunakan bahasa Portugis. Tahap preprocessing meliputi huruf kecil, penghapusan URL, angka, tanda baca, aksen, dan stopword Portugis. Rating dipakai sebagai label awal untuk mengevaluasi apakah pola bahasa dalam teks konsisten dengan kepuasan pelanggan.


In [ ]:
def preprocess_portuguese(text):
    text = unidecode(str(text).lower())
    text = re.sub(r"https?://\S+|www\.\S+", " ", text)
    text = re.sub(r"[^a-z\s]", " ", text)
    tokens = [
        token for token in text.split()
        if len(token) > 2 and token not in PORTUGUESE_STOPWORDS
    ]
    return " ".join(tokens)

text_reviews = integrated[
    integrated["has_review_text"] & integrated["rating_sentiment"].notna()
][["order_id", "review_text", "rating_sentiment", "review_score", "dominant_category"]].copy()

if len(text_reviews) > NLP_SAMPLE_MAX:
    text_reviews = text_reviews.sample(NLP_SAMPLE_MAX, random_state=RANDOM_STATE)

text_reviews["clean_text"] = text_reviews["review_text"].apply(preprocess_portuguese)
text_reviews = text_reviews[text_reviews["clean_text"].str.len() > 2].copy()

print(f"Ulasan teks yang dianalisis: {len(text_reviews):,}")
display(text_reviews[["review_text", "clean_text", "rating_sentiment"]].head(5))


In [ ]:
# Word cloud dipisahkan agar tidak mencampur kata positif dan negatif.
positive_text = " ".join(text_reviews.loc[text_reviews["rating_sentiment"] == "Positif", "clean_text"])
negative_text = " ".join(text_reviews.loc[text_reviews["rating_sentiment"] == "Negatif", "clean_text"])

fig, axes = plt.subplots(1, 2, figsize=(18, 7))
if positive_text.strip():
    wc_pos = WordCloud(width=900, height=500, background_color="white", colormap="Greens").generate(positive_text)
    axes[0].imshow(wc_pos, interpolation="bilinear")
axes[0].set_title("Kata Dominan pada Ulasan Positif")
axes[0].axis("off")

if negative_text.strip():
    wc_neg = WordCloud(width=900, height=500, background_color="white", colormap="Reds").generate(negative_text)
    axes[1].imshow(wc_neg, interpolation="bilinear")
axes[1].set_title("Kata Dominan pada Ulasan Negatif")
axes[1].axis("off")
plt.tight_layout()
plt.show()


In [ ]:
def top_ngrams(text_series, ngram_range=(1, 2), top_n=15):
    vectorizer = TfidfVectorizer(ngram_range=ngram_range, min_df=3, max_features=10_000)
    matrix = vectorizer.fit_transform(text_series)
    scores = np.asarray(matrix.sum(axis=0)).ravel()
    terms = np.asarray(vectorizer.get_feature_names_out())
    order = scores.argsort()[::-1][:top_n]
    return pd.DataFrame({"term": terms[order], "tfidf_score": scores[order]})

top_negative_terms = top_ngrams(
    text_reviews.loc[text_reviews["rating_sentiment"] == "Negatif", "clean_text"],
    ngram_range=(1, 2), top_n=20
)
display(top_negative_terms)

plt.figure(figsize=(10, 6))
sns.barplot(data=top_negative_terms.sort_values("tfidf_score"), x="tfidf_score", y="term")
plt.title("Kata/Frasa Penting dalam Ulasan Negatif")
plt.xlabel("Total TF-IDF")
plt.ylabel("")
plt.tight_layout()
plt.show()


### 10.1 Klasifikasi sentimen teks dengan TF-IDF dan Logistic Regression


In [ ]:
X_text = text_reviews["clean_text"]
y_text = text_reviews["rating_sentiment"]

X_train_text, X_test_text, y_train_text, y_test_text = train_test_split(
    X_text, y_text, test_size=0.20, random_state=RANDOM_STATE, stratify=y_text
)

text_model = Pipeline([
    ("tfidf", TfidfVectorizer(
        ngram_range=(1, 2), min_df=3, max_df=0.95,
        max_features=20_000, sublinear_tf=True
    )),
    ("model", LogisticRegression(
        max_iter=1_000, class_weight="balanced", random_state=RANDOM_STATE
    ))
])

text_model.fit(X_train_text, y_train_text)
pred_text = text_model.predict(X_test_text)

text_accuracy = accuracy_score(y_test_text, pred_text)
text_f1_macro = f1_score(y_test_text, pred_text, average="macro")
print(f"Accuracy : {text_accuracy:.3f}")
print(f"Macro F1 : {text_f1_macro:.3f}\n")
print(classification_report(y_test_text, pred_text, digits=3))

ConfusionMatrixDisplay.from_predictions(
    y_test_text, pred_text,
    labels=["Negatif", "Netral", "Positif"],
    cmap="Blues", xticks_rotation=30
)
plt.title("Confusion Matrix — Klasifikasi Sentimen Teks")
plt.tight_layout()
plt.show()


### 10.2 Topic modeling untuk ulasan negatif


In [ ]:
negative_reviews = text_reviews.loc[
    text_reviews["rating_sentiment"] == "Negatif", "clean_text"
].dropna()

topic_vectorizer = TfidfVectorizer(
    min_df=5, max_df=0.90, ngram_range=(1, 2), max_features=8_000
)
topic_matrix = topic_vectorizer.fit_transform(negative_reviews)
nmf_model = NMF(n_components=N_TOPICS, random_state=RANDOM_STATE, init="nndsvda", max_iter=400)
topic_weights = nmf_model.fit_transform(topic_matrix)
topic_terms = topic_vectorizer.get_feature_names_out()

topic_rows = []
for topic_no, component in enumerate(nmf_model.components_, start=1):
    top_indices = component.argsort()[::-1][:10]
    words = ", ".join(topic_terms[top_indices])
    topic_rows.append({
        "topic": f"Topik {topic_no}",
        "keywords": words,
        "document_count": int((topic_weights.argmax(axis=1) == topic_no - 1).sum())
    })

topics_table = pd.DataFrame(topic_rows).sort_values("document_count", ascending=False)
display(topics_table)


## 11. Predictive analytics: model early-warning ulasan negatif

Model berikut menggunakan data operasional terstruktur untuk memprediksi apakah suatu pesanan berisiko memperoleh rating 1–2. Model tidak memakai `review_score` atau teks ulasan sebagai prediktor sehingga menghindari *target leakage*.

Model ini cocok digunakan setelah status pengiriman diketahui tetapi sebelum/ketika survei kepuasan dikirim.


In [ ]:
model_features = [
    "item_value", "freight_total", "freight_ratio", "item_count",
    "product_count", "seller_count", "payment_installments",
    "delivery_days", "delivery_delay_days", "is_late",
    "dominant_category", "primary_payment_type", "customer_state"
]

model_data = integrated.dropna(subset=["negative_review"]).copy()
X = model_data[model_features]
y = model_data["negative_review"].astype(int)

numeric_features = [
    "item_value", "freight_total", "freight_ratio", "item_count",
    "product_count", "seller_count", "payment_installments",
    "delivery_days", "delivery_delay_days", "is_late"
]
categorical_features = ["dominant_category", "primary_payment_type", "customer_state"]

numeric_pipeline = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler())
])
categorical_pipeline = Pipeline([
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("onehot", OneHotEncoder(handle_unknown="ignore", min_frequency=20))
])

preprocessor = ColumnTransformer([
    ("num", numeric_pipeline, numeric_features),
    ("cat", categorical_pipeline, categorical_features)
])

structured_model = Pipeline([
    ("preprocessor", preprocessor),
    ("model", LogisticRegression(
        max_iter=1_000, class_weight="balanced", random_state=RANDOM_STATE
    ))
])

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.20, random_state=RANDOM_STATE, stratify=y
)
structured_model.fit(X_train, y_train)
pred = structured_model.predict(X_test)
prob = structured_model.predict_proba(X_test)[:, 1]

model_metrics = {
    "accuracy": accuracy_score(y_test, pred),
    "precision": precision_score(y_test, pred, zero_division=0),
    "recall": recall_score(y_test, pred, zero_division=0),
    "f1": f1_score(y_test, pred, zero_division=0),
    "roc_auc": roc_auc_score(y_test, prob),
}
display(pd.DataFrame([model_metrics]).T.rename(columns={0: "score"}).style.format("{:.3f}"))
print(classification_report(y_test, pred, target_names=["Tidak negatif", "Negatif"], digits=3))

ConfusionMatrixDisplay.from_predictions(y_test, pred, cmap="Oranges")
plt.title("Confusion Matrix — Early-warning Ulasan Negatif")
plt.tight_layout()
plt.show()


In [ ]:
# Koefisien Logistic Regression membantu menjelaskan faktor yang menaikkan/menurunkan risiko.
feature_names = structured_model.named_steps["preprocessor"].get_feature_names_out()
coefficients = structured_model.named_steps["model"].coef_[0]
coef_table = pd.DataFrame({"feature": feature_names, "coefficient": coefficients})
coef_table["odds_ratio"] = np.exp(coef_table["coefficient"])

risk_factors = coef_table.sort_values("coefficient", ascending=False).head(15)
protective_factors = coef_table.sort_values("coefficient", ascending=True).head(15)

print("Faktor yang paling meningkatkan risiko ulasan negatif:")
display(risk_factors)
print("Faktor yang paling menurunkan risiko ulasan negatif:")
display(protective_factors)


## 12. Prescriptive analytics dan kesimpulan otomatis

Bagian berikut menerjemahkan hasil aktual menjadi temuan, implikasi, dan tindakan bisnis. Kalimat yang dihasilkan dapat digunakan sebagai dasar pembahasan laporan, tetapi tetap perlu dibaca bersama tabel dan visualisasi sebelumnya.


In [ ]:
# Menentukan kategori bervolume memadai dengan tingkat ulasan negatif tertinggi.
category_action = category_summary.sort_values("negative_rate", ascending=False).head(1)
worst_category = category_action.index[0] if not category_action.empty else "tidak tersedia"
worst_category_negative = category_action["negative_rate"].iloc[0] if not category_action.empty else np.nan

ontime_mean = ontime_scores.mean()
late_mean = late_scores.mean()
late_share = delivered["is_late"].mean()
negative_share = reviewed["negative_review"].mean()
repeat_share = (customer_summary["frequency"] > 1).mean()
top_topic = topics_table.iloc[0]["keywords"] if not topics_table.empty else "tidak tersedia"

decisions = []

if p_mann < 0.05 and late_mean < ontime_mean:
    decisions.append({
        "priority": "Tinggi",
        "finding": f"Pesanan terlambat memiliki rating rata-rata {late_mean:.2f}, dibanding {ontime_mean:.2f} untuk pesanan tepat/lebih awal.",
        "decision": "Terapkan early-warning keterlambatan, notifikasi proaktif, dan kompensasi terarah sebelum survei kepuasan dikirim."
    })
else:
    decisions.append({
        "priority": "Sedang",
        "finding": "Perbedaan rating berdasarkan ketepatan pengiriman belum menunjukkan bukti yang cukup kuat.",
        "decision": "Pantau SLA pengiriman dan evaluasi faktor lain seperti kualitas produk dan kesesuaian deskripsi."
    })

if not category_action.empty:
    decisions.append({
        "priority": "Tinggi",
        "finding": f"Kategori '{worst_category}' memiliki tingkat ulasan negatif {worst_category_negative:.1%} pada kategori dengan volume memadai.",
        "decision": "Audit seller, kualitas produk, kemasan, dan akurasi deskripsi pada kategori tersebut."
    })

decisions.append({
    "priority": "Sedang",
    "finding": f"Topik ulasan negatif paling dominan ditandai kata/frasa: {top_topic}.",
    "decision": "Baca contoh ulasan pada topik dominan, beri label bisnis, lalu tetapkan pemilik tindakan dan target perbaikannya."
})

if model_metrics["roc_auc"] >= 0.70:
    decisions.append({
        "priority": "Tinggi",
        "finding": f"Model early-warning menghasilkan ROC-AUC {model_metrics['roc_auc']:.3f}.",
        "decision": "Uji model pada data periode baru dan gunakan skor risiko untuk memprioritaskan service recovery."
    })
else:
    decisions.append({
        "priority": "Eksperimen",
        "finding": f"ROC-AUC model early-warning sebesar {model_metrics['roc_auc']:.3f}; kemampuan diskriminasi masih perlu ditingkatkan.",
        "decision": "Tambahkan fitur seller, jarak pengiriman, histori pelanggan, dan validasi berbasis waktu sebelum implementasi."
    })

decision_table = pd.DataFrame(decisions)
display(decision_table)


In [ ]:
conclusion = f'''
### Kesimpulan Analisis

Analisis mengintegrasikan **{integrated['order_id'].nunique():,} pesanan** dan
**{integrated['customer_unique_id'].nunique():,} pelanggan unik** pada tingkat transaksi.
Data terstruktur mencakup transaksi, nilai produk, pembayaran, kategori, lokasi, serta
kinerja pengiriman. Data tidak terstruktur berasal dari teks ulasan pelanggan.
Sebanyak **{late_share:.1%}** pesanan delivered tercatat terlambat dan
**{negative_share:.1%}** review berating tergolong negatif. Pelanggan yang melakukan
lebih dari satu pembelian berjumlah **{repeat_share:.1%}** dari pelanggan unik.

Uji Mann–Whitney menghasilkan p-value **{p_mann:.4g}**. Rata-rata rating pesanan
tepat atau lebih awal adalah **{ontime_mean:.2f}**, sedangkan pesanan terlambat
**{late_mean:.2f}**. Korelasi Spearman antara keterlambatan dan rating adalah
**{rho:.3f}** dengan p-value **{p_spearman:.4g}**. Hasil ini harus dibaca bersama
ukuran efek dan konteks operasional, bukan hanya berdasarkan signifikansi statistik.

Model sentimen berbasis teks memperoleh macro-F1 **{text_f1_macro:.3f}**.
Model early-warning berbasis data operasional memperoleh ROC-AUC
**{model_metrics['roc_auc']:.3f}** dan recall kelas negatif
**{model_metrics['recall']:.3f}**. Model belum boleh langsung dipakai dalam produksi;
diperlukan validasi waktu, pemantauan bias, dan evaluasi biaya kesalahan keputusan.

Dari perspektif Data-driven Decision Making, prioritas keputusan adalah meningkatkan
ketepatan pengiriman, mengaudit kategori dan seller dengan risiko tinggi, serta
menggunakan insight teks untuk menjelaskan akar keluhan yang tidak terlihat dalam
angka transaksi. Integrasi kedua jenis data menghasilkan keputusan yang lebih utuh:
data terstruktur menunjukkan **apa yang terjadi**, sedangkan teks ulasan membantu
menjelaskan **mengapa hal tersebut terjadi**.
'''
display(Markdown(conclusion))


## 13. Ekspor hasil analisis

File keluaran:

- `olist_integrated_order_level.csv`: dataset satu baris per transaksi.
- `olist_customer_summary.csv`: ringkasan RFM dan kepuasan pelanggan.
- `olist_decision_recommendations.csv`: rekomendasi keputusan.
- `olist_analysis_summary.xlsx`: workbook berisi ringkasan utama.


In [ ]:
OUTPUT_DIR = Path("/content/olist_analysis_output")
OUTPUT_DIR.mkdir(exist_ok=True)

integrated.to_csv(OUTPUT_DIR / "olist_integrated_order_level.csv", index=False)
customer_summary.to_csv(OUTPUT_DIR / "olist_customer_summary.csv", index=False)
decision_table.to_csv(OUTPUT_DIR / "olist_decision_recommendations.csv", index=False)

with pd.ExcelWriter(OUTPUT_DIR / "olist_analysis_summary.xlsx", engine="openpyxl") as writer:
    pd.DataFrame({"KPI": kpis.keys(), "Nilai": kpis.values()}).to_excel(writer, sheet_name="KPI", index=False)
    category_summary.reset_index().to_excel(writer, sheet_name="Category Summary", index=False)
    stats_table.to_excel(writer, sheet_name="Statistical Tests", index=False)
    topics_table.to_excel(writer, sheet_name="Negative Topics", index=False)
    decision_table.to_excel(writer, sheet_name="Recommendations", index=False)
    customer_summary.head(50_000).to_excel(writer, sheet_name="Customer Summary", index=False)

archive_path = shutil.make_archive("/content/olist_analysis_output", "zip", OUTPUT_DIR)
print("Hasil tersimpan:", archive_path)

# Aktifkan dua baris berikut jika ingin langsung mengunduh ZIP dari Colab.
# from google.colab import files
# files.download(archive_path)


## 14. Keterbatasan dan etika

1. Data bersifat historis (2016–2018) dan berasal dari Brasil sehingga hasil tidak otomatis mewakili e-commerce Indonesia saat ini.
2. Tidak semua pelanggan menulis komentar; analisis teks hanya mewakili pelanggan yang memberikan teks ulasan.
3. Rating dipakai sebagai label sentimen awal. Sarkasme dan makna kontekstual dapat menyebabkan kesalahan klasifikasi.
4. Hubungan statistik tidak selalu membuktikan sebab-akibat.
5. Data pelanggan sudah dianonimkan. Hindari upaya mengidentifikasi individu.
6. Evaluasi model harus mempertimbangkan precision, recall, dampak bisnis, dan biaya intervensi—bukan accuracy saja.

## Referensi data

Olist. *Brazilian E-Commerce Public Dataset by Olist*. Kaggle:  
https://www.kaggle.com/datasets/olistbr/brazilian-ecommerce
